## Advanced — `overlay` & `macvlan` / `ipvlan`

Everything so far was **single-host**: all containers on one daemon, on a bridge. Two drivers step beyond that — you won't reach for them often, but you should know what they're for.

**`overlay` — one network across many hosts.** When containers on host A must reach containers on host B *as if on the same LAN*, you need an overlay. It uses **VXLAN**: each cross-host packet is wrapped in a UDP datagram, sent over the physical network between hosts, and unwrapped on arrival; the hosts gossip about which container lives where so routing stays in sync.

```bash
docker network create -d overlay app-overlay     # inside a Swarm
```

Overlay networks are created **inside a Swarm cluster** (module 09 covers Swarm + overlay end to end). The one-liner to keep: **overlay is how Swarm services on different machines talk** — if you're not in a Swarm, you won't use it.

**`macvlan` / `ipvlan` — a real IP on the physical LAN.** `macvlan` gives the container its own **MAC address** on the host's LAN, so to switches, routers, and DHCP it looks like a wholly separate machine with a real LAN IP and **no NAT**. `ipvlan` is similar but shares the host's MAC (for switches that reject unknown MACs).

```bash
docker network create -d macvlan --subnet=192.168.1.0/24 \
  --gateway=192.168.1.1 -o parent=eth0 lan-net
```

Reach for it only in niche cases — a legacy app that **broadcasts on the LAN** (mDNS, DLNA, old industrial protocols), IoT/home-automation bridges, or packet-capture appliances needing promiscuous mode. For everything else, **`bridge` + `-p` is simpler, more portable, and doesn't depend on your physical network.** (One gotcha: by default the host itself can't reach its own macvlan containers.) That closes networking — bridge for almost everything, and these two for the multi-host and physical-LAN edges.